In [ ]:
import os
import sys
from pathlib import Path

# Set these before importing any `reva` modules. Update the placeholder paths
# to match your machine or shared Jupyter environment.
os.environ["REVA_DATA_ROOT"] = "/path/to/your/data/root"
os.environ["REVA_HF_CACHE_ROOT"] = "/path/to/your/hf_cache/root"
os.environ["REVA_CHECKPOINT_ROOT"] = "/path/to/your/checkpoints/root"
os.environ["REVA_EVAL_RESULTS_ROOT"] = "/path/to/your/eval_results/root"
os.environ["REVA_REGION_DATA_ROOT"] = "/path/to/your/region_data/root"
os.environ["REVA_DECONTAMINATION_ROOT"] = "/path/to/your/decontamination/root"
os.environ["REVA_GROUNDING_DINO_ROOT"] = "/path/to/your/groundingdino/root"
os.environ["REVA_VQAV2_ROOT"] = "/path/to/your/vqav2/root"
os.environ["REVA_TEST_IMAGES_ROOT"] = "/path/to/your/test_images/root"

hf_cache_root = Path(os.environ["REVA_HF_CACHE_ROOT"]).expanduser()
os.environ["HF_HOME"] = str(hf_cache_root)
os.environ["HF_HUB_CACHE"] = str(hf_cache_root / "hub")
os.environ["HF_DATASETS_CACHE"] = str(hf_cache_root / "datasets")
os.environ["TRANSFORMERS_CACHE"] = str(hf_cache_root / "hub")

for var_name in (
    "REVA_DATA_ROOT",
    "REVA_HF_CACHE_ROOT",
    "REVA_CHECKPOINT_ROOT",
    "REVA_EVAL_RESULTS_ROOT",
    "REVA_REGION_DATA_ROOT",
    "REVA_DECONTAMINATION_ROOT",
    "REVA_GROUNDING_DINO_ROOT",
    "REVA_VQAV2_ROOT",
    "REVA_TEST_IMAGES_ROOT",
    "HF_HOME",
    "HF_HUB_CACHE",
    "HF_DATASETS_CACHE",
    "TRANSFORMERS_CACHE",
):
    print(f"{var_name} = {os.environ.get(var_name)}")


def find_reva_project_root(start: Path) -> Path:
    override = os.environ.get("REVA_PROJECT_DIR")
    if override:
        return Path(override).expanduser().resolve()

    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "reva" / "evaluation.py").is_file() and (candidate / "reva" / "config.py").is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate the ReVA project root from the current working directory. "
        "Set REVA_PROJECT_DIR to your cloned repo path."
    )


PROJECT_ROOT = find_reva_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())


# MMBench pHash Decontamination

Remove curriculum samples whose images perceptually overlap **[MMBench](https://github.com/open-compass/MMBench)** evaluation images.

**Reference benchmark:** MMBench ships as TSV files with **base64-encoded images** (official links via [VLMEvalKit](https://github.com/open-compass/VLMEvalKit)). This notebook:
1. Downloads EN **Dev + Test** TSVs (v1.0 and v1.1)
2. Decodes images to `{index}.jpg` per split
3. Runs pHash Hamming ≤ 4 against your curriculum

**Technique:** Same pHash pipeline as POPE / VQAv2 (`dataset.py`).

**Recommended input:** `curriculum_pope_vqav2_clean.pkl` (chain after POPE + VQAv2 decontamination).

**Outputs:**
- `curriculum_vs_mmbench_phash.json`
- `phash_matches_curriculum_mmbench_report.csv`
- `curriculum_eval_clean.pkl` — final training curriculum safe for POPE, VQAv2, and MMBench eval

In [ ]:
!pip install imagehash pillow "numpy<2.0" tqdm matplotlib pandas pyarrow datasets huggingface_hub img2dataset --quiet

## 0. Download MMBench TSV files (run once)

Images are **inside** the TSV (base64). No separate image zip. Total download ~200 MB for the four EN splits below.

**Note:** Use **HTTP** links (official VLMEvalKit README). The host's HTTPS certificate is expired — `wget` exit code 5 / `curl` 60.

Default splits (EN, v1.0 + v1.1):
- `MMBench_DEV_EN`, `MMBench_TEST_EN`
- `MMBench_DEV_EN_V11`, `MMBench_TEST_EN_V11`

In [ ]:
%%bash
set -euo pipefail

MMB_DIR="${REVA_MMBENCH_ROOT:-$HOME/reva-data/mmbench}/tsv"
LOG_DIR="${REVA_DOWNLOAD_LOG_ROOT:-$HOME/reva-data/download_logs}"
mkdir -p "$MMB_DIR" "$LOG_DIR"

step()     { echo "[$(date +%H:%M:%S)] $1"; }
done_msg() { echo "[$(date +%H:%M:%S)] done: $1"; }
skip_msg() { echo "[$(date +%H:%M:%S)] skip: $1"; }

download_tsv() {
  local name="$1" url="$2"
  local dest="$MMB_DIR/${name}.tsv"
  # HTTPS fails (expired cert); official README links use HTTP.
  if [ -f "$dest" ] && [ "$(stat -c%s "$dest" 2>/dev/null || stat -f%z "$dest")" -lt 1000000 ]; then
    rm -f "$dest"
  fi
  if [ ! -f "$dest" ]; then
    step "MMBench ${name} ..."
    wget -c -O "$dest" "$url"
    done_msg "$name"
  else skip_msg "$name"; fi
}

BASE="http://opencompass.openxlab.space/utils/VLMEval"
download_tsv MMBench_DEV_EN       "$BASE/MMBench_DEV_EN.tsv"
download_tsv MMBench_TEST_EN      "$BASE/MMBench_TEST_EN.tsv"
download_tsv MMBench_DEV_EN_V11    "$BASE/MMBench_DEV_EN_V11.tsv"
download_tsv MMBench_TEST_EN_V11  "$BASE/MMBench_TEST_EN_V11.tsv"

echo "[$(date +%H:%M:%S)] MMBench TSV downloads complete under $MMB_DIR"

In [ ]:
import pandas as pd
#from vlmeval.smp.vlm import decode_base64_to_image
import base64
import io
from PIL import Image

df = pd.read_csv(Path(os.environ.get("REVA_MMBENCH_ROOT") or os.path.expanduser("~/reva-data/mmbench")) / "tsv" / "MMBench_DEV_EN.tsv", sep="\t")
print(df.columns.tolist())
print(df.iloc[0]['image'][:50])  # should start with base64-looking characters

# 2. Define the alternative decode function
def decode_base64_to_image(base64_string):
    # Strip potential headers or handle byte inputs safely
    image_bytes = base64.b64decode(base64_string)
    return Image.open(io.BytesIO(image_bytes))

# if 'image' in df.columns:
#     sample_base64 = df['image'].iloc[0]
#     img = decode_base64_to_image(sample_base64)
#     img.show()  

from IPython.display import display

if 'image' in df.columns:
    sample_base64 = df['image'].iloc[0] # Added index [0] to select the first row
    img = decode_base64_to_image(sample_base64)
    display(img)

## 1. Setup

In [ ]:
import os
import json
import csv
import pickle
import random
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
from PIL import Image

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['HF_HOME'] = os.environ.get('REVA_HF_CACHE_ROOT') or os.path.expanduser('~/reva-data/hf_cache')

from reva.config import ProjectionAConfig
from reva.dataset import (
    load_coco_detection_samples,
    load_refcoco_samples,
    load_visual_genome_samples,
    load_grit_samples,
    MMBENCH_TSV_URLS,
    prepare_mmbench_reference_images,
    generate_curriculum_decontamination_log,
    filter_curriculum_from_log,
    HAMMING_THRESH,
)

config = ProjectionAConfig()
DECONTAM_DIR = Path(os.environ.get('REVA_DECONTAMINATION_ROOT') or os.path.expanduser('~/reva-data/decontamination'))
MMBENCH_DIR = Path(os.environ.get('REVA_MMBENCH_ROOT') or os.path.expanduser('~/reva-data/mmbench'))
DECONTAM_DIR.mkdir(parents=True, exist_ok=True)

# EN dev+test, v1.0 and v1.1 — covers standard MMBench leaderboard splits
MMBENCH_SPLITS = list(MMBENCH_TSV_URLS.keys())

VQAV2_CLEAN_PKL = DECONTAM_DIR / 'curriculum_pope_vqav2_clean.pkl'
POPE_CLEAN_PKL = DECONTAM_DIR / 'curriculum_pope_clean.pkl'

print('Config ready.')
print(f'MMBench splits: {MMBENCH_SPLITS}')
print(f'pHash Hamming threshold: {HAMMING_THRESH}')

## 2. Load curriculum (chain from VQAv2 → POPE → full)

Prefer **`curriculum_pope_vqav2_clean.pkl`**, then `curriculum_pope_clean.pkl`, else rebuild full curriculum.

In [ ]:
if VQAV2_CLEAN_PKL.exists():
    with open(VQAV2_CLEAN_PKL, 'rb') as f:
        all_curriculum = pickle.load(f)
    print(f'Loaded VQAv2-clean curriculum: {VQAV2_CLEAN_PKL}')
elif POPE_CLEAN_PKL.exists():
    with open(POPE_CLEAN_PKL, 'rb') as f:
        all_curriculum = pickle.load(f)
    print(f'Loaded POPE-clean curriculum: {POPE_CLEAN_PKL}')
else:
    print('No prior clean pickle — building full curriculum ...')
    coco_samples = load_coco_detection_samples(config.data_dir)
    refcoco_samples = load_refcoco_samples(config.data_dir, splits=['train'])
    vg_samples = load_visual_genome_samples(
        config.data_dir, max_per_image=config.vg_max_annotations_per_image
    )
    grit_samples = load_grit_samples(
        config.data_dir, shard=0, max_images=250000,
        max_boxes_per_image=16, use_ref_exps=True,
        min_clip_l14=0.30, min_box_frac=0.05,
    )
    vg_samples = vg_samples + grit_samples
    all_curriculum = coco_samples + refcoco_samples + vg_samples

print(f'Total curriculum rows: {len(all_curriculum):,}')
print(f'Unique train images: {len({s["image_path"] for s in all_curriculum}):,}')

## 3. Extract MMBench reference images from TSV

Decodes base64 → `mmbench/images/{split}/{index}.jpg`. Run once; cached on disk afterward.

**Note:** VLMEvalKit TSVs deduplicate images — many rows store a short index reference (e.g. `241`) instead of base64. Only ~1,164 unique images are extracted per EN dev split (~4,329 rows total).

In [ ]:
mmbench_ref_paths, mmbench_per_split = prepare_mmbench_reference_images(
    MMBENCH_DIR, split_names=MMBENCH_SPLITS
)

print('\nPer-split image counts:')
for split_name, paths in mmbench_per_split.items():
    print(f'  {split_name:<22} {len(paths):>6,} images')

assert len(mmbench_ref_paths) > 0, 'No MMBench reference images extracted. Check Section 0 downloads.'

## 4. Clear shared pHash cache

Force rebuild of reference matrix from **MMBench** images (not POPE / VQAv2 cache).

In [ ]:
PHASH_CACHE_FILES = [
    Path(os.environ.get('REVA_DATA_ROOT') or os.path.expanduser('~/reva-data')) / 'ref_packed.npy',
    Path(os.environ.get('REVA_DATA_ROOT') or os.path.expanduser('~/reva-data')) / 'train_packed.npy',
    Path(os.environ.get('REVA_DATA_ROOT') or os.path.expanduser('~/reva-data')) / 'train_valid_indices.npy',
]

for p in PHASH_CACHE_FILES:
    if p.exists():
        p.unlink()
        print(f'Deleted cache: {p}')
    else:
        print(f'No cache (ok): {p}')

print('Ready to hash MMBench reference + curriculum from scratch.')

## 5. Run pHash decontamination (MMBench reference)

In [ ]:
LOG_PATH = DECONTAM_DIR / 'curriculum_vs_mmbench_phash.json'

log_path = generate_curriculum_decontamination_log(
    curriculum_samples=all_curriculum,
    all_ref_paths=mmbench_ref_paths,
    config=config,
    output_json_path=str(LOG_PATH),
    type='phash',
)

print(f'Log saved: {log_path}')

## 6. Summarise matches

In [ ]:
with open(log_path) as f:
    log_data = json.load(f)

phash_matches = log_data['phash_matches']
corrupt_indices = set(log_data.get('corrupt_indices', []))
unique_removed = set(m['train_idx'] for m in phash_matches)

print(f'pHash match rows: {len(phash_matches):,}')
print(f'Unique curriculum rows flagged: {len(unique_removed):,} / {len(all_curriculum):,}')
print(f'Corrupt/unreadable rows: {len(corrupt_indices):,}')
print(f'Remaining after pHash purge: {len(all_curriculum) - len(unique_removed):,}')

hamming_dist = Counter(m['hamming_distance'] for m in phash_matches)
print('\nHamming distance distribution:')
for d in sorted(hamming_dist):
    print(f'  distance {d}: {hamming_dist[d]:,}')

In [ ]:
removed_sources = Counter(all_curriculum[idx].get('source', 'unknown') for idx in unique_removed)
total_sources = Counter(s.get('source', 'unknown') for s in all_curriculum)

print(f"{'source':<20} {'total':>10} {'removed':>10} {'remaining':>10} {'% removed':>10}")
print('-' * 65)
for source in sorted(total_sources.keys()):
    total = total_sources[source]
    removed = removed_sources.get(source, 0)
    remaining = total - removed
    pct = removed / total * 100 if total else 0
    print(f'{source:<20} {total:>10,} {removed:>10,} {remaining:>10,} {pct:>9.1f}%')

In [ ]:
ref_triggered = {m['reference_image'] for m in phash_matches}
print(f'MMBench reference images total: {len(mmbench_ref_paths):,}')
print(f'MMBench images with ≥1 pHash match: {len(ref_triggered):,}')
print(f'MMBench images with no match: {len(mmbench_ref_paths) - len(ref_triggered):,}')

for split_name, paths in mmbench_per_split.items():
    split_set = {str(p) for p in paths}
    hit = len(split_set & ref_triggered)
    print(f'  {split_name:<22} {hit:>5,} / {len(paths):,} images hit')

## 7. Visual spot-check (optional)

In [ ]:
def show_phash_match(match):
    train_img = Image.open(match['train_image']).convert('RGB')
    ref_img = Image.open(match['reference_image']).convert('RGB')
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(train_img)
    axes[0].set_title(f"Curriculum [{match['train_idx']}]\n{Path(match['train_image']).name}")
    axes[0].axis('off')
    axes[1].imshow(ref_img)
    axes[1].set_title(f"MMBench ref (Hamming={match['hamming_distance']})\n{Path(match['reference_image']).name}")
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()


if phash_matches:
    exact = [m for m in phash_matches if m['hamming_distance'] == 0]
    borderline = [m for m in phash_matches if m['hamming_distance'] == HAMMING_THRESH]
    print(f'Exact duplicates (distance=0): {len(exact):,}')
    print(f'Borderline (distance={HAMMING_THRESH}): {len(borderline):,}')
    if exact:
        show_phash_match(random.choice(exact))
    if borderline:
        show_phash_match(random.choice(borderline))
else:
    print('No pHash matches — curriculum is clean w.r.t. MMBench.')

## 8. Export CSV audit trail

In [ ]:
csv_path = DECONTAM_DIR / 'phash_matches_curriculum_mmbench_report.csv'
fieldnames = ['train_idx', 'train_image', 'reference_image', 'hamming_distance', 'train_source']

with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for m in phash_matches:
        writer.writerow({
            'train_idx': m['train_idx'],
            'train_image': m['train_image'],
            'reference_image': m['reference_image'],
            'hamming_distance': m['hamming_distance'],
            'train_source': all_curriculum[m['train_idx']].get('source', 'unknown'),
        })

print(f'Saved {len(phash_matches):,} rows -> {csv_path}')

## 9. Save final eval-safe curriculum

In [ ]:
clean_curriculum = filter_curriculum_from_log(
    curriculum_samples=all_curriculum,
    json_log_path=str(log_path),
)

clean_pkl = DECONTAM_DIR / 'curriculum_pope_vqav2_mmbencheval_clean.pkl'
with open(clean_pkl, 'wb') as f:
    pickle.dump(clean_curriculum, f)

print(f'Final eval-safe curriculum: {clean_pkl}')
print(f'Rows: {len(clean_curriculum):,}')
print(f'Unique images: {len({s["image_path"] for s in clean_curriculum}):,}')

## 10. Decontamination chain summary

| Step | Notebook | Output pickle |
|------|----------|---------------|
| 1 | `pope_decontamination.ipynb` | `curriculum_pope_clean.pkl` |
| 2 | `vqav2_decontamination.ipynb` | `curriculum_pope_vqav2_clean.pkl` |
| 3 | `mmbench_decontamination.ipynb` | **`curriculum_pope_vqav2_mmbencheval_clean.pkl`** |

Use **`curriculum_pope_vqav2_mmbencheval_clean.pkl`** for Stage 2 training.

**Stage 3 note:** If you fine-tune on GQA or other data before MMBench eval, run the same pHash pass on that FT corpus separately (GQA overlaps VG).

**Stage 1 note:** If using LLaVA 558K for Projection-B, decontaminate that corpus against MMBench images too before Stage 1 training.

### Training notebook snippet

```python
import pickle

with open(Path(os.environ.get('REVA_DECONTAMINATION_ROOT') or os.path.expanduser('~/reva-data/decontamination')) / 'curriculum_eval_clean.pkl', 'rb') as f:
    all_curriculum = pickle.load(f)

coco_samples = [s for s in all_curriculum if s.get('source') == 'coco']
refcoco_samples = [s for s in all_curriculum if s.get('source') in ('refcoco', 'refcocog', 'refcoco+')]
vg_samples = [s for s in all_curriculum if s.get('source') in ('visual_genome', 'grit')]
```

### MMBench eval (after Stage 3)

Use [VLMEvalKit](https://github.com/open-compass/VLMEvalKit) with `MMBench_DEV_EN` / `MMBench_TEST_EN` (or `_V11` variants). Example:

```bash
python run.py --data MMBench_DEV_EN --model your_model
```